# Notebook 02: Oracle Retraining (`02_oracle_retrain.ipynb`)

### Objectives & Workflow:
1. **Load Persisted Partitions**: Load the exact split artifacts persisted by Notebook 01 for each `seed` (no re-splitting).
2. **Retrain from Scratch on Retain Only (Naive/Oracle)**: Train fresh model (`resnet18` / `vit11m`) using only the retain training set $D_R$ (38,250 samples) using `model_instruction.md` configurations.
3. **Validation Selection**: Use validation set $D_V$ (7,500 samples) for early stopping and checkpoint selection; test set $D_T$ remains untouched until final reporting.
4. **Reference Benchmark**: Compute 3-tier metrics (Output, LP, NCC) on validation (reference for Notebook 04c) and test sets across seeds `[0, 1, ..., 9]`.

In [ ]:
# ============================================================
# SETUP: Clone repo from GitHub & add to Python path
# ============================================================
import sys
import os
import subprocess

REPO_URL = "https://github.com/tiensinh2/Unlearning_experiments_1.git"
REPO_DIR = "Unlearning_experiments_1"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL}...")
    try:
        from git import Repo
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "gitpython", "-q"])
        from git import Repo
    Repo.clone_from(REPO_URL, REPO_DIR)
    print(f"Repository cloned to {REPO_DIR}")
else:
    print(f"Repository already exists at {REPO_DIR}")
    from git import Repo
    repo = Repo(REPO_DIR)
    origin = repo.remotes.origin
    origin.pull()
    print("Pulled latest changes")

repo_path = os.path.abspath(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
    print(f"Added {repo_path} to sys.path")
else:
    print(f"{repo_path} already in sys.path")

In [ ]:
import os
import copy
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

from experiment_utils import (
    build_classifier,
    get_dataset_transforms,
    IndexedDataset,
    compute_output_accuracy,
    evaluate_three_tier_metrics,
    visualize_feature_space_and_boundaries,
    compute_class_mean_geometry,
    save_class_mean_geometry,
    MODEL_CONFIGS
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device}")

## 1. Configurations & Oracle Training Loop (from `model_instruction.md`)

In [ ]:
DATASET_NAME = "cifar10"
SPLIT_SEED = 123  # Single fixed seed for dataset partitioning
MODEL_NAME = "resnet18"  # Options: resnet18, vit11m

# Load hyperparameters matching model_instruction.md
cfg = MODEL_CONFIGS[DATASET_NAME]
MODEL_SEEDS = cfg["model_seeds"]  # from MODEL_CONFIGS[DATASET_NAME]['model_seeds']
BATCH_SIZE = cfg["batch_size"]    # 256
EPOCHS = cfg["num_epochs"]        # 182 for cifar10/100, 50 for mnist/fashion/utkface
LR = cfg["lr"]                    # 0.1
NUM_CLASSES = 10 if DATASET_NAME in ["cifar10", "mnist", "fashion_mnist"] else (100 if DATASET_NAME == "cifar100" else 5)
WEIGHT_DECAY = 5e-4
MOMENTUM = 0.9

train_tf, eval_tf = get_dataset_transforms(DATASET_NAME)
from data_loader import load_raw_dataset as _load_raw_dataset
raw_train, raw_test = _load_raw_dataset(DATASET_NAME, root="./data")
if raw_test is None:
    raise ValueError(f"Dataset '{DATASET_NAME}' has no separate test split via load_raw_dataset; handle UTKFace via build_unlearning_splits instead.")

os.makedirs("./artifacts/checkpoints", exist_ok=True)
os.makedirs("./artifacts/metrics", exist_ok=True)
os.makedirs("./artifacts/splits", exist_ok=True)

# Search path in Kaggle input or local artifacts
kaggle_split = f"/kaggle/input/datasets/kiethe/cifar10-experimentnb1/{DATASET_NAME}_seed_{SPLIT_SEED}_split.json"
local_split = f"./artifacts/splits/{DATASET_NAME}_seed_{SPLIT_SEED}_split.json"
split_file = kaggle_split if os.path.exists(kaggle_split) else local_split
if not os.path.exists(split_file):
    raise FileNotFoundError(f"Split artifact not found at {kaggle_split} or {local_split}. Run Notebook 01 first.")
print(f"Using split artifact: {split_file}")
    
with open(split_file, "r") as f:
    split_info = json.load(f)

# Retain-only training data and evaluation sets
retain_train_set = IndexedDataset(Subset(raw_train, split_info["train_r_indices"]), train_tf)
retain_eval_set = IndexedDataset(Subset(raw_train, split_info["train_r_indices"]), eval_tf)
forget_eval_set = IndexedDataset(Subset(raw_train, split_info["train_f_indices"]), eval_tf)
val_set = IndexedDataset(Subset(raw_train, split_info["val_indices"]), eval_tf)
test_set = IndexedDataset(Subset(raw_test, split_info["test_indices"]), eval_tf)

retain_train_loader = DataLoader(retain_train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
retain_eval_loader = DataLoader(retain_eval_set, batch_size=BATCH_SIZE, shuffle=False)
forget_eval_loader = DataLoader(forget_eval_set, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

for seed in MODEL_SEEDS:
    print(f"\n==================================================")
    print(f"--- Retraining Oracle | Model: {MODEL_NAME} | Seed: {seed} ---")
    print(f"==================================================")
    
    # Build fresh model initialized with seed
    torch.manual_seed(seed)
    oracle_model = build_classifier(model_name=MODEL_NAME, num_classes=NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(oracle_model.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    best_val_acc = 0.0
    best_state = None
    history = []
    
    for epoch in range(1, EPOCHS + 1):
        oracle_model.train()
        total_loss = 0.0
        for images, targets in retain_train_loader:
            images, targets = images.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = oracle_model(images)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * images.size(0)
        scheduler.step()
        
        val_acc = compute_output_accuracy(oracle_model, val_loader, device=device)
        history.append({"epoch": epoch, "train_loss": total_loss / len(retain_train_set), "val_acc": val_acc})
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(oracle_model.state_dict())
            
        if epoch % 10 == 0 or epoch == EPOCHS:
            print(f"Epoch [{epoch:03d}/{EPOCHS:03d}] - Retain Loss: {total_loss / len(retain_train_set):.4f} - Val Acc: {val_acc:.4f} (Best: {best_val_acc:.4f})")
    
    # Restore best oracle checkpoint
    oracle_model.load_state_dict(best_state)
    ckpt_path = f"./artifacts/checkpoints/oracle_{MODEL_NAME}_seed_{seed}.pt"
    torch.save(best_state, ckpt_path)
    print(f"Saved Oracle checkpoint to: {ckpt_path}")
    
    # Evaluate 3-tier metrics
    # Validation metric: output accuracy on the held-out D_V split.
    val_metrics = {"output_val": compute_output_accuracy(oracle_model, val_loader, device=device)}
    test_metrics = evaluate_three_tier_metrics(
        model=oracle_model, probe_train_loader=retain_eval_loader, retain_eval_loader=retain_eval_loader, forget_eval_loader=forget_eval_loader, num_classes=NUM_CLASSES, seed=seed, device=device, test_eval_loader=test_loader, nonmember_loader=val_loader
    )
    
    # Reference geometry: direction vectors global_mean -> class_mean of the Oracle on D_R
    # (forget directions on D_F use the same retain global mean). Used as the standard in 03/04a/04b/04c.
    oracle_geom = compute_class_mean_geometry(
        oracle_model, retain_eval_loader, forget_eval_loader, num_classes=NUM_CLASSES, device=device
    )
    geom_path = f"./artifacts/metrics/oracle_geometry_{MODEL_NAME}_seed_{seed}.npz"
    save_class_mean_geometry(oracle_geom, geom_path)
    retain_dir_norms = np.linalg.norm(oracle_geom["retain_directions"], axis=1)
    print(f"Saved Oracle reference geometry to: {geom_path}")
    print(f"Oracle ||global_mean -> class_mean|| (D_R) per class: {np.round(retain_dir_norms, 4).tolist()}")

    results = {
        "model_type": "oracle",
        "model_name": MODEL_NAME,
        "seed": seed,
        "split_seed": SPLIT_SEED,
        "validation_reference_metrics": val_metrics,
        "test_metrics": test_metrics,
        "oracle_retain_direction_norms": retain_dir_norms.tolist(),
        "training_history": history
    }
    
    res_path = f"./artifacts/metrics/oracle_{MODEL_NAME}_seed_{seed}.json"
    with open(res_path, "w") as f:
        json.dump(results, f, indent=2)
    
    print(f"Oracle Retain (D_R) Output Acc: {test_metrics['output_retain']:.4f} | Forget (D_F) Output Acc: {test_metrics['output_forget']:.4f} | Test (D_T) Acc: {test_metrics.get('output_test', 0.0):.4f}")
    print(f"Oracle Retain (D_R) LP Acc:     {test_metrics['lp_retain']:.4f} | Forget (D_F) LP Acc:     {test_metrics['lp_forget']:.4f} | Test (D_T) LP:  {test_metrics.get('lp_test', 0.0):.4f}")
    print(f"Oracle Retain (D_R) NCC Acc:    {test_metrics['ncc_retain']:.4f} | Forget (D_F) NCC Acc:    {test_metrics['ncc_forget']:.4f} | Test (D_T) NCC: {test_metrics.get('ncc_test', 0.0):.4f}")
    print(f"Saved Oracle metrics to: {res_path}")

    # Feature space visualization (retain classes, forget set, global mean -> class mean arrows) for first seed
    if seed == MODEL_SEEDS[0]:
        os.makedirs("./artifacts/plots", exist_ok=True)
        vis_save = f"./artifacts/plots/features_oracle_{MODEL_NAME}_seed_{seed}.png"
        visualize_feature_space_and_boundaries(
            model=oracle_model,
            retain_loader=retain_eval_loader,
            forget_loader=forget_eval_loader,
            num_classes=NUM_CLASSES,
            title=f"Feature Space: Oracle Retrain on D_R ({MODEL_NAME}) - Retain(D_R) vs Forget(D_F)",
            device=device,
            save_path=vis_save
        )